# Task 07 — MOGA Feasibility and Reproducibility Notebook

> **Notice & Computational Cost Warning**: MOGA (NSGA-II) optimization evaluates optics configurations over multiple generations. This notebook operates independently of the primary `bts.ipynb` workflow and enforces strict physical feasibility.

## 1. Setup & Package Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.nkm.moga import (
    BTSMOGAConfig,
    BTSMOGAProblem,
    run_bts_moga,
    save_moga_results_json
)

In [ ]:
# ── Simulation Configuration Summary ─────────────────────────────────────────
# Prints a table of all simulation parameters: their defaults (from config
# dataclasses) and any values reconfigured explicitly in this notebook.

import sys
from pathlib import Path

_repo_root = Path('..').resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.nkm.moga import BTSMOGAConfig

# ── Defaults (from class definition) ─────────────────────────────────────────
_default = BTSMOGAConfig()

# ── Values set in this notebook (Cell 4: config = BTSMOGAConfig(...)) ────────
_nb_pop_size = 20
_nb_n_gen    = 15
_nb_seed     = 42

# ── Table printer ─────────────────────────────────────────────────────────────
def _flag(default, notebook):
    return '⟵ reconfigured' if default != notebook else ''

rows = [
    # header
    ('Parameter', 'Default', 'This Notebook', 'Unit', 'Note'),
    ('-' * 35, '-' * 18, '-' * 18, '-' * 10, '-' * 20),
    # NSGA-II
    ('pop_size',
     _default.pop_size, _nb_pop_size, 'individuals',
     _flag(_default.pop_size, _nb_pop_size)),
    ('n_gen',
     _default.n_gen, _nb_n_gen, 'generations',
     _flag(_default.n_gen, _nb_n_gen)),
    ('random seed',
     _default.seed, _nb_seed, '—',
     _flag(_default.seed, _nb_seed)),
    # Decision variables
    ('n_quadrupoles',     9, 9, 'count', ''),
    ('quad_bounds (K)',
     f'[{_default.quad_bounds[0]}, {_default.quad_bounds[1]}]',
     f'[{_default.quad_bounds[0]}, {_default.quad_bounds[1]}]',
     'm⁻²', ''),
    # Physics constraints
    ('beta_max_limit',
     _default.beta_max_limit, _default.beta_max_limit, 'm', ''),
    ('mismatch_max_limit',
     _default.mismatch_max_limit, _default.mismatch_max_limit, '—', ''),
    ('aperture_radius',
     f'{_default.aperture_radius_m*1e3:.2f}',
     f'{_default.aperture_radius_m*1e3:.2f}', 'mm', ''),
    ('emittance_x',
     f'{_default.emittance_x_mrad:.0e}',
     f'{_default.emittance_x_mrad:.0e}', 'm·rad', ''),
    ('energy_spread (σ_δ)',
     f'{_default.energy_spread:.2e}',
     f'{_default.energy_spread:.2e}', '—', ''),
    # Objectives
    ('n_objectives',      3, 3, '—', 'f1=mismatch, f2=β_max, f3=disp'),
    ('feasibility_tol',   '1e-5', '1e-5', '—', 'constraint violation ≤ ε'),
    # Re-evaluation
    ('eval_n_mc_seeds',
     _default.eval_n_mc_seeds, _default.eval_n_mc_seeds, 'seeds', ''),
]

col_w = [36, 19, 19, 11, 28]
sep   = '+' + '+'.join('-' * w for w in col_w) + '+'
print()
print('  SIMULATION CONFIGURATION — 03_bts_moga_pareto')
print(sep)
for i, row in enumerate(rows):
    line = '|' + '|'.join(f' {str(v):<{col_w[j]-2}} ' for j, v in enumerate(row)) + '|'
    print(line)
    if i in (0, 1):
        print(sep)
print(sep)
print()


## 2. MOGA Configuration & Optimization Execution

In [ ]:
config = BTSMOGAConfig(
    pop_size=20,
    n_gen=15,
    seed=42
)
print(f"Running NSGA-II MOGA with pop_size={config.pop_size}, n_gen={config.n_gen}, seed={config.seed}...")
result = run_bts_moga(config)
print(f"Optimization completed in {result.runtime_seconds:.2f} seconds.")
print(f"Success: {result.success}, Feasible fraction: {result.feasible_fraction*100:.1f}%")
print(f"Feasible Pareto solutions found: {len(result.pareto_x)}")

## 3. Representative Pareto Solutions

In [ ]:
reps = result.representative_solutions
for name, sol in reps.items():
    print(f"=== {name.upper()} ===")
    print(f"  Total Mismatch (Mx+My): {sol['total_mismatch']:.4f}")
    print(f"  Peak Beta [m]:         {sol['peak_beta']:.4f}")
    print(f"  Residual Disp [m]:      {sol['residual_dispersion']:.4f}\n")

## 4. Result Archival (JSON/CSV)

In [ ]:
output_dir = Path("results/publication_moga")
save_moga_results_json(result, output_dir=output_dir)
print(f"Saved MOGA results to {output_dir}/")